In [ ]:
from typing import Dict, List, Tuple
from .types import DwellEvent, EnergyConfig, MergeCandidate


def schedule_charging_dwells(
    duties: List[List[int]],
    merge_info: Dict[Tuple[int, int], MergeCandidate],
    config: EnergyConfig,
) -> List[DwellEvent]:
    """
    Depot-dwell charging scheduler.

    For each selected block merge i -> j, schedule the required depot charging
    during the dwell between the return from block i and the pull-out for block j.

    This matches the thesis logic:
        block i -> depot -> charge during feasible dwell -> block j

    The merge graph already checks whether the required charging duration fits
    into the available dwell. This function materializes the corresponding
    charging events.
    """

    events: List[DwellEvent] = []
    wh_per_minute = config.charge_rate_wh_per_hour / 60.0

    for duty in duties:
        for i, j in zip(duty, duty[1:]):
            candidate = merge_info.get((i, j))

            if candidate is None:
                continue

            start = candidate.arrive_depot_time
            end = start + candidate.required_charge_minutes * 60

            if end > candidate.latest_depart_time:
                continue

            start_energy = candidate.start_energy
            charged_energy = candidate.required_charge_minutes * wh_per_minute
            end_energy = min(
                config.battery_capacity,
                start_energy + charged_energy,
            )

            events.append(
                DwellEvent(
                    from_block=i,
                    to_block=j,
                    depot_id=candidate.depot_id,
                    start_time=start,
                    end_time=end,
                    start_energy=start_energy,
                    end_energy=end_energy,
                )
            )

    return events